# AI-PAVE: YOLOv8 Pothole Detector GPU Training
### Fast Cloud GPU Training Guide (Google Colab / Kaggle)

This notebook trains the **AI-PAVE Pothole Detection Model (YOLOv8)** using a free NVIDIA T4 GPU.
- **Training Time**: ~15–20 minutes on T4 GPU (compared to 8+ hours on CPU)
- **Output**: `best.pt` model weights ready to download and place into `artifacts/models/pothole_detector_yolo.pt`

## 1. Verify GPU Acceleration
Make sure your runtime is set to **GPU**:
- In Google Colab: **Runtime** -> **Change runtime type** -> **T4 GPU**
- In Kaggle: **Settings** (right panel) -> **Accelerator** -> **GPU T4 x2**

In [ ]:
!nvidia-smi

import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not enabled! Please enable GPU in runtime settings.")

## 2. Install Ultralytics YOLOv8

In [ ]:
!pip install -q ultralytics

## 3. Dataset Setup

### Option A: If using Google Colab with an uploaded `datasets.zip` or Google Drive
Uncomment and run if you uploaded `datasets.zip` directly to Colab:

In [ ]:
# If you uploaded datasets.zip to Colab, extract it:
import os
import zipfile

if os.path.exists('datasets.zip'):
    print("Extracting datasets.zip...")
    with zipfile.ZipFile('datasets.zip', 'r') as zip_ref:
        zip_ref.extractall('datasets')
    print("Extraction complete!")

### Option B: If downloading RDD-2022 from Kaggle directly inside Colab
Uncomment to download and process in under 2 minutes at 100MB/s:

In [ ]:
# To download directly via Kaggle API in Colab:
# 1. Upload your kaggle.json file to Colab
# 2. Uncomment and run:
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d aliabdelmenam/rdd-2022
# !unzip -q rdd-2022.zip -d datasets/raw

## 4. Configure YOLO `potholes.yaml`

In [ ]:
import yaml
import os
from pathlib import Path

dataset_root = os.path.abspath('datasets')

yaml_content = {
    'path': dataset_root,
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'names': {
        0: 'pothole'
    }
}

with open('potholes.yaml', 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

print("Created potholes.yaml:")
with open('potholes.yaml') as f:
    print(f.read())

## 5. Train YOLOv8 on GPU
We use `batch=32` (or `16`) which runs at blazing speed on a 16GB T4 GPU (~40 seconds per epoch).

In [ ]:
from ultralytics import YOLO

# Initialize pretrained YOLOv8 nano detector
model = YOLO('yolov8n.pt')

# Train for 15-20 epochs (sufficient for convergence on RDD pothole data)
results = model.train(
    data='potholes.yaml',
    epochs=15,
    imgsz=640,
    batch=32,
    workers=4,
    device=0,
    project='AI_PAVE_runs',
    name='pothole_detector',
    patience=5,
    save=True,
    exist_ok=True
)

print("Training Complete!")

## 6. Validate on Test Set & View Metrics

In [ ]:
# Evaluate on held-out test split
metrics = model.val(data='potholes.yaml', split='test')
print("mAP@50:", metrics.box.map50)
print("mAP@50-95:", metrics.box.map)

## 7. Download `best.pt` Weights
Run this cell to download the trained weights file directly to your computer.

In [ ]:
best_weights = 'AI_PAVE_runs/pothole_detector/weights/best.pt'

try:
    from google.colab import files
    files.download(best_weights)
    print("Download initiated in browser!")
except Exception as e:
    print(f"Weights saved at: {best_weights}")
    print("On Kaggle: Check the 'Output' pane on the right side to download best.pt")

## 8. Put `best.pt` into your AI-PAVE Local Project
After downloading `best.pt`:
1. Copy `best.pt` to: `E:\AI-PAVE\artifacts\models\pothole_detector_yolo.pt`
2. Start your backend API server:
   ```bash
   python -m uvicorn app.main:app --host 0.0.0.0 --port 8000 --reload
   ```
3. Your software will now detect real potholes using your real trained model!